# Clase 9 · Notebook 01
# Del texto a los números: tokenización y embeddings

**Introducción al Deep Learning — Módulo 4, Unidad 1: Redes recurrentes y NLP**

Las redes no entienden texto, solo números. En este cuaderno veremos las dos formas de representar palabras:

1. **Tokenizar**: partir el texto en palabras y asignarles un número.
2. **One-hot / Bag of Words**: representación dispersa basada en conteo.
3. **Embeddings**: vectores densos que capturan el **significado**.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Tokenización: del texto a una secuencia de números

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
tf.random.set_seed(42); np.random.seed(42)

frases = [
    "el gato duerme en el sofa",
    "el perro corre en el parque",
    "un gato y un perro juegan",
]

# TextVectorization aprende un vocabulario y convierte cada frase en una secuencia de enteros
vectorizar = tf.keras.layers.TextVectorization(output_mode="int", output_sequence_length=6)
vectorizar.adapt(frases)

print("Vocabulario aprendido:")
for i, palabra in enumerate(vectorizar.get_vocabulary()):
    print(f"  {i}: {palabra!r}")
print("\n'el gato duerme en el sofa' ->", vectorizar(["el gato duerme en el sofa"]).numpy()[0])

Cada palabra recibe un número (token). Las posiciones 0 y 1 se reservan para relleno y palabras
desconocidas. La frase se convierte en una secuencia de enteros de longitud fija.

## 2. One-hot / Bag of Words: conteo de palabras

Con `output_mode="multi_hot"` obtenemos, para cada frase, un vector que marca qué palabras del vocabulario
aparecen. Es la idea de **Bag of Words**: ignora el orden y produce vectores grandes y dispersos.

In [ ]:
bow = tf.keras.layers.TextVectorization(output_mode="multi_hot")
bow.adapt(frases)
vocab = bow.get_vocabulary()
print("Tamaño del vocabulario:", len(vocab))
vec = bow(["el gato duerme en el sofa"]).numpy()[0]
print("Vector BOW (1 = la palabra aparece):")
print(vec.astype(int))
print("Palabras presentes:", [vocab[i] for i, v in enumerate(vec) if v == 1])

Problema de BOW: el vector es **disperso** (casi todo ceros) y no refleja que 'gato' y 'perro' son más
parecidos entre sí que 'gato' y 'sofá'. No captura el **significado**.

## 3. Embeddings: vectores densos con significado

Una capa `Embedding` asigna a cada palabra un **vector denso** que la red **aprende**. Palabras con
significado parecido acaban con vectores cercanos.

In [ ]:
vocab_size = len(vectorizar.get_vocabulary())
DIM = 4   # cada palabra será un vector de 4 números (en la práctica, 50-300)

embedding = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=DIM)

secuencia = vectorizar(["el gato duerme en el sofa"])
vectores = embedding(secuencia)
print("Forma:", vectores.shape, " (1 frase, 6 palabras, 4 dimensiones por palabra)")
print("\nVector (embedding) de la 2ª palabra ('gato'):")
print(vectores.numpy()[0, 1].round(3))

Estos vectores empiezan aleatorios, pero **se ajustan durante el entrenamiento** de la red (como cualquier
otro peso), de modo que acaban codificando relaciones de significado.

## Resumen

- **Tokenizar**: convertir el texto en una secuencia de números (un id por palabra).
- **One-hot / BOW**: vectores dispersos por conteo; ignoran orden y similitud.
- **Embeddings**: vectores densos que capturan el significado; la capa `Embedding` los aprende.

➡️ En el **Notebook 02** usaremos `Embedding + LSTM` para analizar el **sentimiento** de un texto.